# Add miRNA to Merged Datasets

Adds miRNA MB-selected features to each of the 8 existing Clinical+RNA+CNV+MUT+PROT+METH merged files.

**Restart kernel before running.**

**Input:** `MERGE/01_ultra_conservative.csv` ... `08_composite.csv`  
**Output:** same files updated with `MIRNA_` columns + `merge_summary_7modalities.csv`

## Setup — Paths, Dataset Pairs & Best miRNA Config

In [3]:
"""
ADD MIRNA: Add miRNA features to existing Clinical+RNA+CNV+MUT+PROT+METH merged datasets.

Script location: 01_Causal_feature_extraction/
Input/Output:    01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

MIRNA_DIR   = SCRIPT_DIR / "miRNA"
FILT_DIR   = MIRNA_DIR / "statistical_filtered"
MB_DIR     = MIRNA_DIR / "mb_results"
MERGE_DIR  = SCRIPT_DIR / "MERGE"

print(f"Script dir:   {SCRIPT_DIR}")
print(f"miRNA dir: {MIRNA_DIR.exists()} -> {MIRNA_DIR}")
print(f"miRNA MB:  {MB_DIR.exists()} -> {MB_DIR}")
print(f"MERGE dir:    {MERGE_DIR.exists()}")

DATASET_PAIRS = {
    "01_ultra_conservative": "mirna_1_ultra_conservative_50mirnas",
    "02_conservative":       "mirna_2_conservative_50mirnas",
    "03_standard":           "mirna_3_standard_50mirnas",
    "04_fdr_significant":    "mirna_4_fdr_significant_200mirnas",
    "05_balanced":           "mirna_5_balanced_50mirnas",
    "06_correlation":        "mirna_6_correlation_50mirnas",
    "07_top_correlated":     "mirna_7_top_correlated_200mirnas",
    "08_composite":          "mirna_8_composite_200mirnas",
}

# ---------------------------------------------------------------------------
# STEP 1: LOAD PROTEINS SUMMARY AND SELECT BEST CONFIG PER DATASET
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 1: SELECT BEST METHYLATION CONFIG PER DATASET (highest C-index)")
print("="*70)
print(f"Reading: {MB_DIR / 'summary_all_results.csv'}")

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")
print(f"Summary shape: {summary.shape}")
print(f"Datasets in summary: {summary['dataset'].unique().tolist()}")

best_mirna = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index"]]

print(best_mirna.to_string(index=False))

Script dir:   C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
miRNA dir: True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA
miRNA MB:  True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA\mb_results
MERGE dir:    True

STEP 1: SELECT BEST METHYLATION CONFIG PER DATASET (highest C-index)
Reading: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\miRNA\mb_results\summary_all_results.csv
Summary shape: (56, 15)
Datasets in summary: ['mirna_1_ultra_conservative_50mirnas', 'mirna_2_conservative_50mirnas', 'mirna_3_standard_50mirnas', 'mirna_4_fdr_significant_200mirnas', 'mirna_5_balanced_50mirnas', 'mirna_6_correlation_50mirnas', 'mirna_7_top_correlated_200mirnas', 'mirna_8_composite_200mirnas']
                            dataset algorithm  alpha  c_index
   mirna_7_top_correlated_200mirnas      IAMB   0.05 0.584154
       mirna_6_correlation_50mirnas      GSMB   0.10 0.581344
          mirna_5_balanced_50mirnas     

## Helper Functions

In [5]:
# ---------------------------------------------------------------------------
# STEP 2: HELPERS
# ---------------------------------------------------------------------------
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = MB_DIR / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found: {MB_DIR / dataset / f'{algorithm}_alpha{alpha:.2f}_genes.txt'}"
    )


def load_mirna(mirna_dataset, algorithm, alpha):
    candidates = list(FILT_DIR.glob(f"{mirna_dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No miRNA file found for '{mirna_dataset}'")
    mirna_file = candidates[0]

    genes      = load_genes(mirna_dataset, algorithm, alpha)
    prot       = pd.read_csv(mirna_file, index_col=0)
    available  = [g for g in genes if g in prot.columns]
    missing_g  = [g for g in genes if g not in prot.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} miRNA features not in file")
    if not available:
        raise ValueError(f"No miRNA available in {mirna_file.name}")
    prot = prot[available].copy()
    print(f"  miRNA loaded: {prot.shape}  ({mirna_file.name})")

    tumor_mask = prot.index.str.contains(r"-01[A-Z]?$", regex=True)
    prot = prot[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(prot)} samples")

    prot.index = [normalize_id(i) for i in prot.index]

    if prot.index.duplicated().any():
        n_pts = prot.index[prot.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate samples")
        prot = prot.groupby(prot.index).mean()

    prot.columns = [f"MIRNA_{c}" for c in prot.columns]
    return prot

print("\nHelper functions ready")


Helper functions ready


## Add miRNA to All 8 Datasets

In [7]:
# ---------------------------------------------------------------------------
# STEP 3: ADD MIRNA TO ALL 8 DATASETS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: ADD MIRNA TO ALL 8 MERGED DATASETS")
print("="*70)

results = []

for short_name, mirna_dataset in DATASET_PAIRS.items():
    merged_file = MERGE_DIR / f"{short_name}.csv"
    if not merged_file.exists():
        print(f"\nSkipping {short_name} — file not found")
        continue

    row = best_mirna[best_mirna["dataset"] == mirna_dataset]
    if row.empty:
        print(f"\nNo miRNA summary entry for {mirna_dataset} — skipping")
        continue
    row = row.iloc[0]

    print(f"\n{'─'*70}")
    print(f"[{short_name}]")
    print(f"  miRNA dataset: {mirna_dataset}")
    print(f"  Config: {row['algorithm']}  alpha={row['alpha']}  C-index={row['c_index']:.4f}")

    try:
        merged = pd.read_csv(merged_file, index_col=0)

        # drop any MIRNA_ columns from a previous run
        mirna_cols_existing = [c for c in merged.columns if c.startswith("MIRNA_")]
        if mirna_cols_existing:
            merged = merged.drop(columns=mirna_cols_existing)
            print(f"  Dropped {len(mirna_cols_existing)} existing MIRNA_ columns")

        n_clin = sum(1 for c in merged.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in merged.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in merged.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in merged.columns if c.startswith("MUT_"))
        print(f"  Existing: {merged.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  MUT_={n_mut}")

        prot = load_mirna(mirna_dataset, row["algorithm"], float(row["alpha"]))

        common = sorted(set(merged.index) & set(prot.index))
        print(f"  Common patients: {len(common)}  "
              f"(merged={len(merged)}, proteins={len(prot)})")
        if not common:
            raise ValueError("No common patients")

        final = pd.concat([merged.loc[common], prot.loc[common]], axis=1)

        assert final.isna().sum().sum() == 0, \
            f"Missing values: {final.isna().sum().sum()}"

        n_mirna = sum(1 for c in final.columns if c.startswith("MIRNA_"))
        n_clin = sum(1 for c in final.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in final.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in final.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in final.columns if c.startswith("MUT_"))
        print(f"  Final: {final.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  "
              f"MUT_={n_mut}  MIRNA_={n_mirna}  | no missing")

        final.to_csv(merged_file)
        print(f"  Saved: {merged_file.name}")

        results.append({
            "file":             short_name + ".csv",
            "n_samples":        final.shape[0],
            "n_clin":           n_clin,
            "n_rna":            n_rna,
            "n_cnv":            n_cnv,
            "n_mut":            n_mut,
            "n_mirna":           n_mirna,
            "n_total":          final.shape[1],
            "mirna_dataset":     mirna_dataset,
            "mirna_algorithm":   row["algorithm"],
            "mirna_alpha":       row["alpha"],
            "mirna_c_index":     row["c_index"],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


STEP 3: ADD MIRNA TO ALL 8 MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  miRNA dataset: mirna_1_ultra_conservative_50mirnas
  Config: MMMB  alpha=0.05  C-index=0.5077
  Existing: (533, 394)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  miRNA loaded: (1056, 50)  (mirna_1_ultra_conservative_50mirnas.csv)
  After Primary Tumor filter: 1054 samples
  Common patients: 526  (merged=533, proteins=1054)
  Final: (526, 444)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50  MIRNA_=50  | no missing
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  miRNA dataset: mirna_2_conservative_50mirnas
  Config: GSMB  alpha=0.1  C-index=0.5150
  Existing: (533, 394)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  miRNA loaded: (1056, 50)  (mirna_2_conservative_50mirnas.csv)
  After Primary Tumor filter: 1054 samples
  Common patients: 526  (merged=533, proteins=1054)
  Final: (526, 444)  

## Summary

In [9]:
# ---------------------------------------------------------------------------
# STEP 4: SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(MERGE_DIR / "merge_summary_7modalities.csv", index=False)
    print(f"\nCLIN_ same:  {df['n_clin'].nunique()==1}  ({df['n_clin'].iloc[0]})")
    print(f"RNA_ varies: {df['n_rna'].nunique()>1}  ({df['n_rna'].min()}–{df['n_rna'].max()})")
    print(f"CNV_ varies: {df['n_cnv'].nunique()>1}  ({df['n_cnv'].min()}–{df['n_cnv'].max()})")
    print(f"MUT_ varies: {df['n_mut'].nunique()>1}  ({df['n_mut'].min()}–{df['n_mut'].max()})")
    print(f"MIRNA_ varies:{df['n_mirna'].nunique()>1}  ({df['n_mirna'].min()}–{df['n_mirna'].max()})")
    print(f"Total:        {df['n_total'].min()}–{df['n_total'].max()} features")
    print(f"Samples:      {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nSaved: {MERGE_DIR / 'merge_summary_7modalities.csv'}")
else:
    print("No datasets processed successfully.")


STEP 4: SUMMARY
                     file  n_samples  n_clin  n_rna  n_cnv  n_mut  n_mirna  n_total                       mirna_dataset mirna_algorithm  mirna_alpha  mirna_c_index
01_ultra_conservative.csv        526     144     50     50     50       50      444 mirna_1_ultra_conservative_50mirnas            MMMB         0.05       0.507669
      02_conservative.csv        526     144     50     50     50       50      444       mirna_2_conservative_50mirnas            GSMB         0.10       0.515000
          03_standard.csv        526     144     50     50     50       50      444           mirna_3_standard_50mirnas            GSMB         0.20       0.489207
   04_fdr_significant.csv        526     144     50     50     50       50      444   mirna_4_fdr_significant_200mirnas            MMMB         0.05       0.550134
          05_balanced.csv        526     144     50     50     50       50      444           mirna_5_balanced_50mirnas            GSMB         0.20       0.580386